In [1]:
!pip install -q langchain-hub langchain-core requests 

ERROR: Could not find a version that satisfies the requirement langchain-hub (from versions: none)

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for langchain-hub


In [1]:
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import requests 

e:\Lang_Chain\langchain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


create tool

In [2]:
@tool
def multiply(a:int, b:int) -> int:
    """Multiply two integers."""
    return a * b

In [3]:
result = multiply.invoke({"a": 5, "b": 3})
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two integers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


Bind LLM

In [4]:
llm = ChatGroq(model="openai/gpt-oss-120b")
#llm_with_tools = llm.bind_tools([multiply])
llm_with_tools = llm.bind_tools([multiply])

messages = [HumanMessage(content="can you multiply 3 with 4")]

ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)

print(ai_message.tool_calls)

[{'name': 'multiply', 'args': {'a': 3, 'b': 4}, 'id': 'fc_32660ea2-9239-455d-b522-810a493a8dc0', 'type': 'tool_call'}]


In [8]:
print(ai_message)

content='' additional_kwargs={'reasoning_content': 'We need to multiply 3 and 4. Use the function multiply.', 'tool_calls': [{'id': 'fc_32660ea2-9239-455d-b522-810a493a8dc0', 'function': {'arguments': '{"a":3,"b":4}', 'name': 'multiply'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 128, 'total_tokens': 177, 'completion_time': 0.102599824, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.026111542, 'prompt_tokens_details': None, 'queue_time': 0.317314913, 'total_time': 0.128711366}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_3166198c1d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0542d-cd3c-7f71-bf94-ae160ba340a9-0' tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 4}, 'id': 'fc_32660ea2-9239-455d-b522-810a493a8dc0', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 128, 'output_tokens

Create Human Message

In [7]:
tool_result =  multiply.invoke(ai_message.tool_calls[0])
print(tool_result)

content='12' name='multiply' tool_call_id='fc_32660ea2-9239-455d-b522-810a493a8dc0'


In [9]:
messages.append(tool_result)
llm_with_tools.invoke(messages).content

'The product of\u202f3\u202fand\u202f4\u202fis\u202f12.'

In [10]:
print(messages)

[HumanMessage(content='can you multiply 3 with 4', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to multiply 3 and 4. Use the function multiply.', 'tool_calls': [{'id': 'fc_32660ea2-9239-455d-b522-810a493a8dc0', 'function': {'arguments': '{"a":3,"b":4}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 128, 'total_tokens': 177, 'completion_time': 0.102599824, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.026111542, 'prompt_tokens_details': None, 'queue_time': 0.317314913, 'total_time': 0.128711366}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_3166198c1d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0542d-cd3c-7f71-bf94-ae160ba340a9-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 4}, 'id': 'fc_32660ea2-9239-455d-b522-